# Linear Regression

In [20]:
import numpy as np
import sklearn as sk

## 正规方程

In [21]:
class Normalequations:
    def __init__(self, fit_intercept = True):
        self.fit_intercept = fit_intercept
        self.coef_ = None
        self.intercept_ = None

    def _add_intercept(self, X):
        X = np.asarray(X, dtype = float)
        n_samples = X.shape[0]
        ones = np.ones((n_samples, 1))
        X_b = np.c_[ones, X]
        return X_b

    def fit(self, X, y):
        X = np.asarray(X, dtype = float)
        y = np.asarray(y, dtype = float).ravel()
        n_samples, n_features = X.shape
        if y.shape[0] != n_samples:
            raise ValueError("Number of samples in X and y do not match.")
        if self.fit_intercept:
            X_b = self._add_intercept(X)
        else:
            X_b = X

        self.theta_ = np.linalg.pinv(X_b) @ y # pinv是求伪逆的函数
        if self.fit_intercept:
            self.intercept_ = self.theta_[0]
            self.coef_ = self.theta_[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = self.theta_
        return self

    def predict(self, X):
        if self.theta_ is None:
            raise ValueError("模型还没有训练，请先调用fit(X, y)")
        X = np.asarray(X, dtype = float)
        if self.fit_intercept:
            X_b = self._add_intercept(X)
        else:
            X_b = X

        y_pred = X_b @ self.theta_
        return y_pred

    def mse(self, X, y):
        # 均方误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean((y - y_pred) ** 2)
    def rmse(self, X, y):
        # 均方根误差
        return np.sqrt(self.mse(X, y))
    def mae(self, X, y):
        # 评价绝对误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean(np.abs(y - y_pred))
    def score(self, X, y):
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        sse = np.sum((y - y_pred) ** 2)
        sst = np.sum((y - np.mean(y)) ** 2)
        if sst == 0:
            return np.nan

        r2 = 1 - sse / sst

        return r2

## 梯度下降

In [22]:
class GradientDescent:
    def __init__(self, eta = 0.01, max_iter = 1000, tol = 1e-8, fit_intercept = True,
                 random_state = 0, verbose = False, print_every = 100):
        """
        eta:学习率，控制每次参数更新的步长。
        max_iter:最大迭代次数。
        tol:收敛阈值。如果相邻两次损失函数的变化小于tol，则提前停止。
        fit_intercept:是否训练偏置项 b。
        random_state:随机种子，用来初始化参数。
        verbose :是否打印训练过程。
        print_every:每隔多少次迭代打印一次训练信息。
        """
        self.eta = eta
        self.max_iter = max_iter
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.random_state = random_state
        self.verbose = verbose
        self.print_every = print_every

        self.coef_ = None
        self.intercept_ = None
        self.theta_ = None # 完整的参数
        self.n_iter_ = 0 # 实际迭代次数
        self.loss_history_ = [] # 保存每次迭代的损失

    def _add_intercept(self, X):
        # 给 X 前面添加一列 1，用来吸收偏置项 b
        X = np.asarray(X, dtype = float)
        n_samples = X.shape[0]
        ones = np.ones((n_samples, 1))
        X_b = np.c_[ones, X]
        return X_b

    def _compute_loss(self, X_b, y):
        m = X_b.shape[0]
        y_pred = X_b @ self.theta_
        loss = np.sum((y_pred - y) ** 2) / (2 * m)
        return loss

    def fit(self, X, y):
        X = np.asarray(X, dtype = float)
        y = np.asarray(y, dtype = float).ravel()
        n_samples, n_features = X.shape
        if y.shape[0] != n_samples:
            raise ValueError(
                f"X 和 y 的样本数量不一致：X 有 {n_samples} 个样本，但 y 有 {y.shape[0]} 个标签。"
            )
        if self.fit_intercept:
            X_b = self._add_intercept(X)
            n_params = n_features + 1
        else:
            X_b = X
            n_params = n_features

        rng = np.random.default_rng(self.random_state)
        self.theta_ = rng.normal(loc = 0.0, scale = 0.01, size = n_params)
        # 如果不训练偏置项，那么 theta_ 中没有 b
        # 如果训练偏置项，那么 theta_[0] 是 b，theta_[1:] 是 w
        for iteration in range(1, self.max_iter + 1):
            y_pred = X_b @ self.theta_
            error = y_pred - y
            gradient = (X_b.T @ error) / n_samples
            old_theta = self.theta_.copy()
            self.theta_ = self.theta_ - self.eta * gradient
            current_loss = self._compute_loss(X_b, y)
            self.loss_history_.append(current_loss)
            if self.verbose and (iteration == 1 or iteration % self.print_every == 0):
                print(
                    f"Iteration {iteration:05d} | "
                    f"loss = {current_loss:.10f} | "
                    f"gradient_norm = {np.linalg.norm(gradient):.10f}"
                )
            w_change = np.linalg.norm(old_theta - self.theta_)
            if w_change < self.tol:
                self.n_tol_ = iteration
                if self.verbose:
                    print(
                        f"训练提前停止：第 {iteration} 次迭代时，"
                        f"损失变化 {w_change:.10e} 小于 tol = {self.tol}"
                    )

                break
        else:
            self.n_iter_ = self.max_iter
            if self.verbose:
                print("达到最大迭代次数，训练停止。")

        if self.fit_intercept:
            self.intercept_ = self.theta_[0]
            self.coef_ = self.theta_[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = self.theta_
        return self

    def predict(self, X):
        if self.theta_ is None:
            raise ValueError("模型还没有训练，请先调用fit(X, y)")
        X = np.asarray(X, dtype = float)
        if self.fit_intercept:
            X_b = self._add_intercept(X)
        else:
            X_b = X

        y_pred = X_b @ self.theta_
        return y_pred

    def mse(self, X, y):
        # 均方误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean((y - y_pred) ** 2)
    def rmse(self, X, y):
        # 均方根误差
        return np.sqrt(self.mse(X, y))
    def mae(self, X, y):
        # 评价绝对误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean(np.abs(y - y_pred))
    def score(self, X, y):
        # R^2决定系数
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        sse = np.sum((y - y_pred) ** 2)
        sst = np.sum((y - np.mean(y)) ** 2)
        if sst == 0:
            return np.nan

        r2 = 1 - sse / sst

        return r2

## 正规+L2正则项

In [23]:
class Normalequations_L2:
    def __init__(self, fit_intercept = True, lambda1 = 0.01):
        self.fit_intercept = fit_intercept
        self.coef_ = None
        self.intercept_ = None
        self.lambda1 = lambda1

    def _add_intercept(self, X):
        X = np.asarray(X, dtype = float)
        n_samples = X.shape[0]
        ones = np.ones((n_samples, 1))
        X_b = np.c_[ones, X]
        return X_b

    def fit(self, X, y):
        X = np.asarray(X, dtype = float)
        y = np.asarray(y, dtype = float).ravel()
        n_samples, n_features = X.shape
        if y.shape[0] != n_samples:
            raise ValueError("Number of samples in X and y do not match.")
        if self.fit_intercept:
            X_b = self._add_intercept(X)
        else:
            X_b = X
        n_params = X_b.shape[1]
        regular_matrix = n_samples * self.lambda1 * np.eye(n_params) # 为了和梯度下降的λ保持一致
        regular_matrix[0, 0] = 0.0 # 一般不对截距项做正则化
        A = X_b.T @ X_b + regular_matrix
        c = X_b.T @ y
        self.theta_ = np.linalg.solve(A, c)
        if self.fit_intercept:
            self.intercept_ = self.theta_[0]
            self.coef_ = self.theta_[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = self.theta_
        return self

    def predict(self, X):
        if self.theta_ is None:
            raise ValueError("模型还没有训练，请先调用fit(X, y)")
        X = np.asarray(X, dtype = float)
        if self.fit_intercept:
            X_b = self._add_intercept(X)
        else:
            X_b = X

        y_pred = X_b @ self.theta_
        return y_pred

    def mse(self, X, y):
        # 均方误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean((y - y_pred) ** 2)
    def rmse(self, X, y):
        # 均方根误差
        return np.sqrt(self.mse(X, y))
    def mae(self, X, y):
        # 评价绝对误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean(np.abs(y - y_pred))
    def score(self, X, y):
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        sse = np.sum((y - y_pred) ** 2)
        sst = np.sum((y - np.mean(y)) ** 2)
        if sst == 0:
            return np.nan

        r2 = 1 - sse / sst

        return r2

## 梯度下降+L1

In [24]:
class GradientDescent_L1:
    def __init__(self, eta = 0.01, lambda1 = 0.01, max_iter = 1000, tol = 1e-8, fit_intercept = True,
                 random_state = 0, verbose = False, print_every = 100):
        """
        eta:学习率，控制每次参数更新的步长。
        lambda1: 惩罚系数
        max_iter:最大迭代次数。
        tol:收敛阈值。如果相邻两次损失函数的变化小于tol，则提前停止。
        fit_intercept:是否训练偏置项 b。
        random_state:随机种子，用来初始化参数。
        verbose :是否打印训练过程。
        print_every:每隔多少次迭代打印一次训练信息。
        """
        self.eta = eta
        self.lambda1 = lambda1
        self.max_iter = max_iter
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.random_state = random_state
        self.verbose = verbose
        self.print_every = print_every

        self.coef_ = None
        self.intercept_ = None
        self.theta_ = None # 完整的参数
        self.n_iter_ = 0 # 实际迭代次数
        self.loss_history_ = [] # 保存每次迭代的损失

    def _add_intercept(self, X):
        # 给 X 前面添加一列 1，用来吸收偏置项 b
        X = np.asarray(X, dtype = float)
        n_samples = X.shape[0]
        ones = np.ones((n_samples, 1))
        X_b = np.c_[ones, X]
        return X_b

    def _compute_loss(self, X_b, y):
        m = X_b.shape[0]
        y_pred = X_b @ self.theta_
        if self.fit_intercept:
            theta_reg = self.theta_[1:]
        else:
            theta_reg = self.theta_

        loss = np.sum((y_pred - y) ** 2) / (2 * m) + self.lambda1 * np.abs(theta_reg).sum()
        return loss

    def fit(self, X, y):
        X = np.asarray(X, dtype = float)
        y = np.asarray(y, dtype = float).ravel()
        n_samples, n_features = X.shape
        if y.shape[0] != n_samples:
            raise ValueError(
                f"X 和 y 的样本数量不一致：X 有 {n_samples} 个样本，但 y 有 {y.shape[0]} 个标签。"
            )
        if self.fit_intercept:
            X_b = self._add_intercept(X)
            n_params = n_features + 1
        else:
            X_b = X
            n_params = n_features

        rng = np.random.default_rng(self.random_state)
        self.theta_ = rng.normal(loc = 0.0, scale = 0.01, size = n_params)
        # 如果不训练偏置项，那么 theta_ 中没有 b
        # 如果训练偏置项，那么 theta_[0] 是 b，theta_[1:] 是 w
        for iteration in range(1, self.max_iter + 1):
            y_pred = X_b @ self.theta_
            error = y_pred - y
            lambda2 = self.lambda1 * np.sign(self.theta_)
            if self.fit_intercept:
                lambda2[0] = 0.0
            gradient = (X_b.T @ error) / n_samples +lambda2
            old_theta = self.theta_.copy()
            self.theta_ = self.theta_ - self.eta * gradient
            current_loss = self._compute_loss(X_b, y)
            self.loss_history_.append(current_loss)
            if self.verbose and (iteration == 1 or iteration % self.print_every == 0):
                print(
                    f"Iteration {iteration:05d} | "
                    f"loss = {current_loss:.10f} | "
                    f"gradient_norm = {np.linalg.norm(gradient):.10f}"
                )
            theta_change = np.linalg.norm(old_theta - self.theta_)
            if theta_change < self.tol:
                self.n_tol_ = iteration
                if self.verbose:
                    print(
                        f"训练提前停止：第 {iteration} 次迭代时，"
                        f"损失变化 {theta_change:.10e} 小于 tol = {self.tol}"
                    )

                break
        else:
            self.n_iter_ = self.max_iter
            if self.verbose:
                print("达到最大迭代次数，训练停止。")

        if self.fit_intercept:
            self.intercept_ = self.theta_[0]
            self.coef_ = self.theta_[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = self.theta_
        return self

    def predict(self, X):
        if self.theta_ is None:
            raise ValueError("模型还没有训练，请先调用fit(X, y)")
        X = np.asarray(X, dtype = float)
        if self.fit_intercept:
            X_b = self._add_intercept(X)
        else:
            X_b = X

        y_pred = X_b @ self.theta_
        return y_pred

    def mse(self, X, y):
        # 均方误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean((y - y_pred) ** 2)
    def rmse(self, X, y):
        # 均方根误差
        return np.sqrt(self.mse(X, y))
    def mae(self, X, y):
        # 评价绝对误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean(np.abs(y - y_pred))
    def score(self, X, y):
        # R^2决定系数
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        sse = np.sum((y - y_pred) ** 2)
        sst = np.sum((y - np.mean(y)) ** 2)
        if sst == 0:
            return np.nan

        r2 = 1 - sse / sst

        return r2

## 梯度下降+L2

In [25]:
class GradientDescent_L2:
    def __init__(self, eta = 0.01, lambda1 = 0.01, max_iter = 1000, tol = 1e-8, fit_intercept = True,
                 random_state = 0, verbose = False, print_every = 100):
        """
        eta:学习率，控制每次参数更新的步长。
        lambda1: 惩罚系数
        max_iter:最大迭代次数。
        tol:收敛阈值。如果相邻两次损失函数的变化小于tol，则提前停止。
        fit_intercept:是否训练偏置项 b。
        random_state:随机种子，用来初始化参数。
        verbose :是否打印训练过程。
        print_every:每隔多少次迭代打印一次训练信息。
        """
        self.eta = eta
        self.lambda1 = lambda1
        self.max_iter = max_iter
        self.tol = tol
        self.fit_intercept = fit_intercept
        self.random_state = random_state
        self.verbose = verbose
        self.print_every = print_every

        self.coef_ = None
        self.intercept_ = None
        self.theta_ = None # 完整的参数
        self.n_iter_ = 0 # 实际迭代次数
        self.loss_history_ = [] # 保存每次迭代的损失

    def _add_intercept(self, X):
        # 给 X 前面添加一列 1，用来吸收偏置项 b
        X = np.asarray(X, dtype = float)
        n_samples = X.shape[0]
        ones = np.ones((n_samples, 1))
        X_b = np.c_[ones, X]
        return X_b

    def _compute_loss(self, X_b, y):
        m = X_b.shape[0]
        y_pred = X_b @ self.theta_
        loss = np.sum((y_pred - y) ** 2) / (2 * m) + self.lambda1 * np.sum(self.theta_ ** 2) / 2
        return loss

    def fit(self, X, y):
        X = np.asarray(X, dtype = float)
        y = np.asarray(y, dtype = float).ravel()
        n_samples, n_features = X.shape
        if y.shape[0] != n_samples:
            raise ValueError(
                f"X 和 y 的样本数量不一致：X 有 {n_samples} 个样本，但 y 有 {y.shape[0]} 个标签。"
            )
        if self.fit_intercept:
            X_b = self._add_intercept(X)
            n_params = n_features + 1
        else:
            X_b = X
            n_params = n_features

        rng = np.random.default_rng(self.random_state)
        self.theta_ = rng.normal(loc = 0.0, scale = 0.01, size = n_params)
        # 如果不训练偏置项，那么 theta_ 中没有 b
        # 如果训练偏置项，那么 theta_[0] 是 b，theta_[1:] 是 w
        for iteration in range(1, self.max_iter + 1):
            y_pred = X_b @ self.theta_
            error = y_pred - y
            lambda2 = self.lambda1 * self.theta_
            if self.fit_intercept:
                lambda2[0] = 0.0
            gradient = (X_b.T @ error) / n_samples + lambda2
            old_theta = self.theta_.copy()
            self.theta_ = self.theta_ - self.eta * gradient
            current_loss = self._compute_loss(X_b, y)
            self.loss_history_.append(current_loss)
            if self.verbose and (iteration == 1 or iteration % self.print_every == 0):
                print(
                    f"Iteration {iteration:05d} | "
                    f"loss = {current_loss:.10f} | "
                    f"gradient_norm = {np.linalg.norm(gradient):.10f}"
                )
            w_change = np.linalg.norm(old_theta - self.theta_)
            if w_change < self.tol:
                self.n_tol_ = iteration
                if self.verbose:
                    print(
                        f"训练提前停止：第 {iteration} 次迭代时，"
                        f"损失变化 {w_change:.10e} 小于 tol = {self.tol}"
                    )

                break
        else:
            self.n_iter_ = self.max_iter
            if self.verbose:
                print("达到最大迭代次数，训练停止。")

        if self.fit_intercept:
            self.intercept_ = self.theta_[0]
            self.coef_ = self.theta_[1:]
        else:
            self.intercept_ = 0.0
            self.coef_ = self.theta_
        return self

    def predict(self, X):
        if self.theta_ is None:
            raise ValueError("模型还没有训练，请先调用fit(X, y)")
        X = np.asarray(X, dtype = float)
        if self.fit_intercept:
            X_b = self._add_intercept(X)
        else:
            X_b = X

        y_pred = X_b @ self.theta_
        return y_pred

    def mse(self, X, y):
        # 均方误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean((y - y_pred) ** 2)
    def rmse(self, X, y):
        # 均方根误差
        return np.sqrt(self.mse(X, y))
    def mae(self, X, y):
        # 评价绝对误差
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        return np.mean(np.abs(y - y_pred))
    def score(self, X, y):
        # R^2决定系数
        y = np.asarray(y, dtype = float).ravel()
        y_pred = self.predict(X)
        sse = np.sum((y - y_pred) ** 2)
        sst = np.sum((y - np.mean(y)) ** 2)
        if sst == 0:
            return np.nan

        r2 = 1 - sse / sst

        return r2

## 生成数据

In [26]:
rng = np.random.default_rng(42)

n_samples = 200
n_features = 3

# 真实参数
w_true = np.array([2.0, -3.5, 1.2])
b_true = 4.0

# 生成输入数据
X_train = rng.normal(0, 1, size=(n_samples, n_features))

# 生成噪声
noise = rng.normal(0, 0.5, size=n_samples)

# 生成输出标签
y_train = X_train @ w_true + b_true + noise

print("X_train.shape =", X_train.shape)
print("y_train.shape =", y_train.shape)
print("真实 w_true =", w_true)
print("真实 b_true =", b_true)

X_train.shape = (200, 3)
y_train.shape = (200,)
真实 w_true = [ 2.  -3.5  1.2]
真实 b_true = 4.0


## 开始训练

In [27]:
model1 = Normalequations(fit_intercept = True)
model1.fit(X_train, y_train)

print("=" * 80)
print("正规方程线性回归训练结果")
print("=" * 80)
print(f"训练得到的 w = {np.round(model1.coef_, 6)}")
print(f"训练得到的 b = {model1.intercept_:.6f}")
print()

print(f"真实 w_true = {w_true}")
print(f"真实 b_true = {b_true}")
print()

print(f"MSE  = {model1.mse(X_train, y_train):.6f}")
print(f"RMSE = {model1.rmse(X_train, y_train):.6f}")
print(f"MAE  = {model1.mae(X_train, y_train):.6f}")
print(f"R^2  = {model1.score(X_train, y_train):.6f}")

正规方程线性回归训练结果
训练得到的 w = [ 2.018395 -3.503797  1.174169]
训练得到的 b = 3.981472

真实 w_true = [ 2.  -3.5  1.2]
真实 b_true = 4.0

MSE  = 0.256591
RMSE = 0.506548
MAE  = 0.410582
R^2  = 0.983491


In [28]:
model2 = GradientDescent(eta = 0.05, max_iter = 1000, tol = 1e-10,
                           fit_intercept = True, random_state = 0, verbose = False, print_every = 100)
model2.fit(X_train, y_train)
print()
print("=" * 80)
print("梯度下降线性回归训练结果")
print("=" * 80)
print(f"实际迭代次数 n_iter_ = {model2.n_iter_}")
print(f"训练得到的 w = {np.round(model2.coef_, 6)}")
print(f"训练得到的 b = {model2.intercept_:.6f}")
print()
print(f"真实 w_true = {w_true}")
print(f"真实 b_true = {b_true}")
print()
print(f"MSE  = {model2.mse(X_train, y_train):.6f}")
print(f"RMSE = {model2.rmse(X_train, y_train):.6f}")
print(f"MAE  = {model2.mae(X_train, y_train):.6f}")
print(f"R^2  = {model2.score(X_train, y_train):.6f}")


梯度下降线性回归训练结果
实际迭代次数 n_iter_ = 0
训练得到的 w = [ 2.018395 -3.503797  1.174169]
训练得到的 b = 3.981472

真实 w_true = [ 2.  -3.5  1.2]
真实 b_true = 4.0

MSE  = 0.256591
RMSE = 0.506548
MAE  = 0.410582
R^2  = 0.983491


In [29]:
model3 = Normalequations_L2(fit_intercept = True, lambda1 = 0.01)
model3.fit(X_train, y_train)

print("=" * 80)
print("正规方程线性回归训练结果")
print("=" * 80)
print(f"训练得到的 w = {np.round(model3.coef_, 6)}")
print(f"训练得到的 b = {model3.intercept_:.6f}")
print()

print(f"真实 w_true = {w_true}")
print(f"真实 b_true = {b_true}")
print()

print(f"MSE  = {model3.mse(X_train, y_train):.6f}")
print(f"RMSE = {model3.rmse(X_train, y_train):.6f}")
print(f"MAE  = {model3.mae(X_train, y_train):.6f}")
print(f"R^2  = {model3.score(X_train, y_train):.6f}")

正规方程线性回归训练结果
训练得到的 w = [ 1.995404 -3.464501  1.156113]
训练得到的 b = 3.984350

真实 w_true = [ 2.  -3.5  1.2]
真实 b_true = 4.0

MSE  = 0.258620
RMSE = 0.508547
MAE  = 0.412047
R^2  = 0.983360


In [30]:
model4 = GradientDescent_L1(eta = 0.05, lambda1 = 0.01, max_iter = 1000, tol = 1e-10,
                           fit_intercept = True, random_state = 0, verbose = False, print_every = 100)
model4.fit(X_train, y_train)
print()
print("=" * 80)
print("梯度下降线性回归训练结果")
print("=" * 80)
print(f"实际迭代次数 n_iter_ = {model4.n_iter_}")
print(f"训练得到的 w = {np.round(model4.coef_, 6)}")
print(f"训练得到的 b = {model4.intercept_:.6f}")
print()
print(f"真实 w_true = {w_true}")
print(f"真实 b_true = {b_true}")
print()
print(f"MSE  = {model4.mse(X_train, y_train):.6f}")
print(f"RMSE = {model4.rmse(X_train, y_train):.6f}")
print(f"MAE  = {model4.mae(X_train, y_train):.6f}")
print(f"R^2  = {model4.score(X_train, y_train):.6f}")


梯度下降线性回归训练结果
实际迭代次数 n_iter_ = 0
训练得到的 w = [ 2.006934 -3.491633  1.160906]
训练得到的 b = 3.982354

真实 w_true = [ 2.  -3.5  1.2]
真实 b_true = 4.0

MSE  = 0.256960
RMSE = 0.506912
MAE  = 0.411313
R^2  = 0.983467


In [31]:
model5 = GradientDescent_L2(eta = 0.05, lambda1 = 0.01, max_iter = 1000, tol = 1e-10,
                           fit_intercept = True, random_state = 0, verbose = False, print_every = 100)
model5.fit(X_train, y_train)
print()
print("=" * 80)
print("梯度下降线性回归训练结果")
print("=" * 80)
print(f"实际迭代次数 n_iter_ = {model5.n_iter_}")
print(f"训练得到的 w = {np.round(model5.coef_, 6)}")
print(f"训练得到的 b = {model5.intercept_:.6f}")
print()
print(f"真实 w_true = {w_true}")
print(f"真实 b_true = {b_true}")
print()
print(f"MSE  = {model5.mse(X_train, y_train):.6f}")
print(f"RMSE = {model5.rmse(X_train, y_train):.6f}")
print(f"MAE  = {model5.mae(X_train, y_train):.6f}")
print(f"R^2  = {model5.score(X_train, y_train):.6f}")


梯度下降线性回归训练结果
实际迭代次数 n_iter_ = 0
训练得到的 w = [ 1.995404 -3.464501  1.156113]
训练得到的 b = 3.984350

真实 w_true = [ 2.  -3.5  1.2]
真实 b_true = 4.0

MSE  = 0.258620
RMSE = 0.508547
MAE  = 0.412047
R^2  = 0.983360
